# 安装
首先，使用pip以下命令安装 Ragas：

```shell
pip install ragas
```

如果您想尝试最新的功能，请从主分支安装最新版本：
```shell
pip install git+https://github.com/explodinggradients/ragas.git
```

如果您打算贡献并修改代码，请确保克隆存储库并将其设置为可编辑的安装。

```shell
git clone https://github.com/explodinggradients/ragas.git 
cd ragas 
pip install -e .
```


## 评估简单的LLM应用
本指南旨在说明使用 测试和评估 LLM 应用程序的简单工作流程ragas。本指南要求您具备 AI 应用程序构建和评估方面的最低知识水平。请参阅我们的ragas[安装说明](https://docs.ragas.io/en/stable/getstarted/install/)进行安装。

## 评估
在本指南中，您将评估文本摘要流程。目标是确保输出摘要准确捕捉文本中指定的所有关键细节，例如增长数据、市场洞察和其他重要信息。

ragas提供了多种用于分析 LLM 应用程序性能的方法，这些方法称为[指标](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)。每个指标都需要一组预定义的数据点，并使用这些数据点来计算指示性能的分数

### 使用Non-LLM指标进行评估

这是一个使用`BleuScore`分数进行分数汇总的简单示例

In [1]:
from ragas import SingleTurnSample
from ragas.metrics import BleuScore

test_data = {
    "user_input": "summarise given text\nThe company reported an 8% rise in Q3 2024, driven by strong performance in the Asian market. Sales in this region have significantly contributed to the overall growth. Analysts attribute this success to strategic marketing and product localization. The positive trend in the Asian market is expected to continue into the next quarter.",
    "response": "The company experienced an 8% increase in Q3 2024, largely due to effective marketing strategies and product adaptation, with expectations of continued growth in the coming quarter.",
    "reference": "The company reported an 8% growth in Q3 2024, primarily driven by strong sales in the Asian market, attributed to strategic marketing and localized products, with continued growth anticipated in the next quarter."
}
metric = BleuScore()
test_data = SingleTurnSample(**test_data)
metric.single_turn_score(test_data)

/opt/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.13718598426177148

## sacreBLEU包参数
- 主要参数    
    - sys_stream    
        - 类型: 字符串列表或文件对象    
        - 作用: 待评测的模型输出（即候选文本，hypothesis）。
        - 示例: ["This is a test.", "Another example."] 或 open("hyp.txt")。
    - ref_streams
        - 类型: 字符串列表的列表 或 文件对象列表
        - 作用: 参考译文（reference），支持多组参考译文。
        - 示例: [["This is a test.", "This is an example."]] 或 [open("ref1.txt"), open("ref2.txt")]。
    - smooth_method
        - 类型: 字符串
        - 作用: 指定平滑方法（避免 0-gram 匹配时的分数为 0）。
        - 可选值:
            - "none"（不平滑）
            - "floor"（默认，添加 0.1 的微小值）
            - "add-k"（加 k 平滑，需配合 smooth_value）
            - "exp"（指数平滑）
    - smooth_value
        - 类型: 浮点数
        - 作用: 平滑时的附加值（如 add-k 中的 k）。
        - 默认: 1.0（当 smooth_method="add-k" 时）。
    - tokenize
        - 类型: 字符串或 None
        - 作用: 指定分词方式（与参考一致以公平比较）。
        - 可选值:
            - "13a"（默认，标准 WMT 分词）
            - "intl"（国际化分词）
            - "zh"（中文分词）
            - "ja-mecab"（日语 MeCab 分词）
            - None（不分词，直接输入需已分词）
    - lowercase
        - 类型: 布尔值
        - 作用: 是否将文本转为小写后计算。
        - 默认: False。
    - force   
        - 类型: 布尔值
        - 作用: 即使存在可疑操作（如混合分词方式），也强制计算。
        - 默认: False。
    - use_effective_order
        - 类型: 布尔值
        - 作用: 使用有效 n-gram 阶数（避免短句的高阶 n-gram 惩罚）。
        - 默认: False。
    - signature
        - 类型: 字符串
        - 作用: 指定 BLEU 的评测标准（如 "nrefs|1"）。一般自动生成，无需手动设置。

## 使用基于 LLM 的指标进行评估

安装 langchain-openai 包

```shell
pip install langchain-openai
```
确保您的 OpenAI 密钥已准备好并在您的环境中可用。
```shell
import os
os.environ["OPENAI_API_KEY"] = "your-openai-key"
```


将 LLM 包装起来，LangchainLLMWrapper以便可以与 ragas 一起使用。

In [5]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/a1-6/Documents/projects/DL/.env")

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o", api_key=os.environ["api_key"], base_url=os.environ["base_url"]))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(api_key=os.environ["api_key"], base_url=os.environ["base_url"]))

In [7]:
from ragas import SingleTurnSample
from ragas.metrics import AspectCritic

test_data = {
    "user_input": "summarise given text\nThe company reported an 8% rise in Q3 2024, driven by strong performance in the Asian market. Sales in this region have significantly contributed to the overall growth. Analysts attribute this success to strategic marketing and product localization. The positive trend in the Asian market is expected to continue into the next quarter.",
    "response": "The company experienced an 8% increase in Q3 2024, largely due to effective marketing strategies and product adaptation, with expectations of continued growth in the coming quarter.",
}

metric = AspectCritic(name="summary_accuracy",llm=evaluator_llm, definition="Verify if the summary is accurate.")
test_data = SingleTurnSample(**test_data)
await metric.single_turn_ascore(test_data)

0

成功！1 表示通过，0 表示失败

## 在数据集上进行评估
在上面的示例中，我们仅使用了一个样本来评估我们的应用程序。然而，仅使用一个样本进行评估并不足以保证结果的可靠性。为了确保评估的可靠性，您应该在测试数据中添加更多测试样本。

在这里，我们将从 Hugging Face Hub 加载数据集，但您可以从任何来源加载数据，例如生产日志或其他数据集。只需确保每个样本包含所选指标的所有必需属性即可。

在我们的例子中，所需的属性是：
- user_input：提供给应用程序的输入（这里是输入文本报告）。
- response：应用程序生成的输出（这里是生成的摘要）。

In [10]:
from datasets import Dataset

dataset = [
    {
        "user_input": "summarise given text\nThe Q2 earnings report revealed a significant 15% increase in revenue, ...",
        "response": "The Q2 earnings report showed a 15% revenue increase, ...",
    },
    {
        "user_input": "summarise given text\nIn 2023, North American sales experienced a 5% decline, ...",
        "response": "Companies are strategizing to adapt to market challenges and ...",
    }
]
eval_dataset = Dataset.from_list(dataset, split="train")

eval_dataset

Dataset({
    features: ['user_input', 'response'],
    num_rows: 2
})

In [11]:
# from datasets import load_dataset
# eval_dataset = load_dataset("explodinggradients/earning_report_summary",split="train")
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_hf_dataset(eval_dataset)
print("Features in dataset:", eval_dataset.features())
print("Total samples in dataset:", len(eval_dataset))

Features in dataset: ['user_input', 'response']
Total samples in dataset: 2


使用数据集进行评估

In [12]:
from ragas import evaluate

results = evaluate(eval_dataset, metrics=[metric])
results

Evaluating: 100%|██████████| 2/2 [00:02<00:00,  1.14s/it]


{'summary_accuracy': 0.5000}

将样本级别分数导出到 Pandas 数据框

In [13]:
results.to_pandas()

,user_input,response,summary_accuracy
0,summarise given text\nThe Q2 earnings report r...,The Q2 earnings report showed a 15% revenue in...,1
1,"summarise given text\nIn 2023, North American ...",Companies are strategizing to adapt to market ...,0
